In [2]:
!pip install python-docx openpyxl
# 安装ngrok
!pip install -q openai streamlit pandas pyngrok
!wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xvzf ngrok-v3-stable-linux-amd64.tgz
!mv ngrok /usr/local/bin/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.1 MB/s eta 0:00:00
ngrok


In [ ]:
# 杀死所有正在运行的 streamlit 和 ngrok 进程
!pkill streamlit
!pkill npx

# 清理可能存在的端口占用
!fuser -k 8501/tcp

In [24]:
# @title
%%writefile storyboard_agent.py
import openai
import streamlit as st
import json
import re
from typing import List, Dict
import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.oxml.ns import qn
import io
import requests
import random

# ===================== Configuration =====================
API_KEY = "sk-vViYVIiKm8Ovmt0Nkbxd12Cs2XCxeo3EUBxD3Zdq8IUJ03HB"
BASE_URL = "https://api.chatanywhere.tech/v1"
AGENT_TYPE = "gpt-3.5-turbo"

IMAGE_API_KEY = "ark-fcf2ff0d-5dbd-4380-a5cc-0f212069c4e7-9fbe8"
IMAGE_BASE_URL = "https://ark.cn-beijing.volces.com/api/v3"
IMAGE_MODEL = "doubao-seedream-4-0-250828"



# ===================== 视觉风格注入 =====================
def apply_ultra_tech_theme():
    st.markdown("""
    <link href="https://fonts.googleapis.com/css2?family=Cormorant+Garamond:ital,wght@0,300;0,400;0,500;0,600;0,700;1,400;1,500&family=Work+Sans:wght@300;400;500;600;700&family=JetBrains+Mono:wght@300;400;500;700&family=Noto+Serif+SC:wght@400;600;700;900&display=swap" rel="stylesheet">
    <style>
        /* ====== 1. 隐藏 Streamlit 默认元素 ====== */
        header {
            background-color: #F4F1EB !important;
            background-image: url('https://images.unsplash.com/photo-1737276746515-19ea864a6b80?w=1920&q=60') !important;
            background-size: cover !important;
            background-attachment: fixed !important;
            background-blend-mode: soft-light !important;
            visibility: visible !important;
        }}
        footer {visibility: hidden !important;}
        #MainMenu {visibility: hidden !important;}
        .stAppDeployButton {display:none !important;}

        /* ====== 2. 全局：文学纸感背景 ====== */
        .stApp {
            background-color: #F4F1EB !important;
            background-image: url('https://images.unsplash.com/photo-1737276746515-19ea864a6b80?w=1920&q=60') !important;
            background-size: cover !important;
            background-attachment: fixed !important;
            background-blend-mode: soft-light !important;
            color: #1C1A17 !important;
            font-family: 'Work Sans', 'Noto Serif SC', sans-serif !important;
        }
        /* 侧边栏：固定宽度，不挤压 */
        [data-testid="stSidebar"] {
            width: 300px !important;
            background-color: #F9F8F5 !important;
            border-right: 1px solid #DCD7CC !important;
        }

        /* 主内容区：自动占满剩余空间 */
        [data-testid="stSidebar"] + div {
            flex-grow: 1 !important; /* 关键：让主内容区占据所有剩余空间 */
            min-width: 0 !important; /* 防止内容溢出 */
        }

        /* 内容容器：取消固定边距，改为内边距 */
        .block-container {
            max-width: 1200px !important;
            margin: 0 auto !important;
            padding-top: 60px !important;
            padding-left: 20rem !important;
            padding-right: 20rem !important;
        }

        /* ====== 3. 侧边栏样式 ====== */
        [data-testid="stSidebarNav"] li {
            margin-bottom: 8px !important;
        }
        [data-testid="stSidebarNav"] span {
            font-family: 'JetBrains Mono', monospace !important;
            font-size: 30px !important;
            font-weight: 500 !important;
            letter-spacing: 0.15em !important;
            text-transform: uppercase !important;
            color: #000000 !important;
            transition: color 0.2s ease !important;
        }
        [data-testid="stSidebarNav"] li:hover span {
            color: #D9381E !important;
        }
        [data-testid="stSidebarNav"] [aria-selected="true"] span {
            color: #D9381E !important;
        }
        /* ====== 3. 衬线大标题 (Cormorant Garamond) ====== */
        h1, h2, h3 {
            font-family: 'Cormorant Garamond', 'Noto Serif SC', serif !important;
            color: #1C1A17 !important;
            text-transform: none !important;
            letter-spacing: -0.03em;
            text-shadow: none !important;
            margin-bottom: 20px !important;
        }
        h1 {
            font-size: 3.2rem !important;
            font-weight: 300 !important;
            line-height: 0.95 !important;
        }
        h2 {
            font-size: 2.5rem !important;
            font-weight: 400 !important;
            letter-spacing: -0.02em;
        }
        h3 {
            font-size: 2.5rem !important;
            font-weight: 600 !important;
        }
        /* ====== 5. 院线风卡片 (替代 tech-card) ====== */
        .tech-card {
            background: #F9F8F5;
            border: 1px solid #DCD7CC;
            padding: 24px;
            margin: 12px 0;
            border-radius: 0px !important;
            transition: transform 0.3s ease, box-shadow 0.3s ease;
            box-shadow: none;
        }
        .tech-card:hover {
            transform: translateY(-2px);
            box-shadow: 0 8px 30px rgba(28, 26, 23, 0.08);
        }
        .tech-card h3 {
            font-family: 'Cormorant Garamond', serif !important;
            color: #1C1A17 !important;
            font-weight: 600 !important;
            text-shadow: none !important;
        }
        .tech-card p, .tech-card b {
            color: rgba(28, 26, 23, 0.7) !important;
            font-weight: 500;
        }

        /* ====== 6. 输入框：手稿风 ====== */
        .stTextArea textarea {
            background-color: #F9F8F5 !important;
            color: #1C1A17 !important;
            border: 1px solid #DCD7CC !important;
            border-radius: 0px !important;
            font-family: 'Work Sans', 'Noto Serif SC', sans-serif !important;
            font-size: 25px !important;
            line-height: 1.8 !important;
            padding: 16px !important;
        }
        .stTextArea textarea:focus {
            border-color: #1C1A17 !important;
            box-shadow: 0 0 0 1px #1C1A17 !important;
        }
        .stTextArea textarea::placeholder {
            color: #6B6862 !important;
            font-style: italic !important;
            font-size: 25px !important;  /* 这里控制 placeholder 文字大小 */
            opacity: 0.6 !important;
        }

        /* ====== 7. 主按钮：深炭电影红 ====== */
        button[data-testid="stBaseButton-primary"] {
            font-weight: 600 !important;
            padding: 20px 32px !important;   /* 上下20px 左右32px */
            border-radius: 12px !important;
            background: linear-gradient(135deg, #ff4d4f 0%, #ff7875 100%) !important; /* 红色渐变背景 */
            border: none !important;
            transition: all 0.3s ease !important;
            box-shadow: 0 4px 12px rgba(255, 77, 79, 0.2) !important; /* 红色阴影 */
        }

        /* 按钮内部文字（强制覆盖嵌套层级） */
        button[data-testid="stBaseButton-primary"] * {
            font-size: 28px !important;
        }

        /* 按钮悬停效果（鼠标放上去时） */
        button[data-testid="stBaseButton-primary"]:hover {
            transform: translateY(-2px) !important; /* 向上移动2px */
            box-shadow: 0 6px 16px rgba(255, 77, 79, 0.3) !important; /* 阴影增强 */
            background: linear-gradient(135deg, #ff7875 0%, #ff4d4f 100%) !important; /* 渐变反转 */
        }

        /* ====== 8. 布局间距 ====== */
        .block-container {
            max-width: 100% !important;
            padding-top: 60px !important;
            padding-left: 2em !important;   /* 关键：取消所有左边距 */
            padding-right: 2em !important;
        }

        /* ====== 9. 下拉选择框 ====== */
        div[data-testid="stSelectbox"] div[data-baseweb="select"] > div {
            width: 100% !important;          /* 宽度100%撑满 */
            height: 100% !important;          /* 高度80%（可按需调整） */
            padding: 12px 16px !important;   /* 内框内边距 */
            margin: 0 !important;             /* 彻底消除内外框间距 */
        }

        /* 下拉框选中文字（覆盖所有嵌套层级，100%生效） */
        div[data-testid="stSelectbox"] div[data-baseweb="select"] *,
        div[data-testid="stSelectbox"] div[data-baseweb="select"] input,
        div[data-testid="stSelectbox"] div[data-baseweb="select"] span,
        div[data-testid="stSelectbox"] div[data-baseweb="select"] div,
        div[data-testid="stSelectbox"] div[role="combobox"] * {
            font-size: 25px !important;
            line-height: 1.8 !important;      /* 防止文字改大后被截断 */
        }

        /* 下拉展开后选项文字（少量/较少/一般...） */
        div[data-baseweb="popover"] li *,
        div[data-baseweb="popover"] li span,
        div[data-baseweb="popover"] li div {
            font-size: 25px !important;
            line-height: 2.5 !important;
        }

        /* 下拉框右侧箭头大小 */
        div[data-testid="stSelectbox"] svg {
            width: 20px !important;
            height: 20px !important;
        }
        /* ====== 10. 折叠框 (Expander) - 分镜面板 ====== */
        .stExpander {
          background: #ffffff !important;
          border-radius: 18px !important;
          box-shadow: 0 4px 20px rgba(0, 0, 0, 0.08) !important; /* 柔和黑阴影 */
          margin: 1px 0 !important;
          padding: 0 !important;
        }

        /* 折叠框摘要行高（增加点击区域） */
        [data-testid="stExpander"] summary {
            line-height: 3.5 !important;
        }

        /* 折叠框摘要文字（📽️ 镜头 X | 类型 | 时长） */
        [data-testid="stExpander"] summary,
        [data-testid="stExpander"] summary * {
            font-size: 32px !important;
            font-weight: 500 !important;
        }

        /* 折叠框摘要布局（左对齐） */
        [data-testid="stExpander"] summary > span {
            text-align: left !important;
            flex: none !important; /* 防止被拉伸 */
        }

        /* 折叠框内正文文字 */
        .stExpander [data-testid="stMarkdown"] p {
            font-size: 28px !important;
            line-height: 1.6 !important;
            color: #1f2937 !important;
            margin: 8px 0 !important;
        }

        /* 折叠框内代码块文字 */
        .stExpander .stCode code {
            font-size: 20px !important;
            line-height: 2 !important;
        }

        .stExpander div[data-testid="stTextInput"] label,
        .stExpander div[data-testid="stTextInput"] label p,
        .stExpander div[data-testid="stTextInput"] label span {
            font-size: 24px !important; /* 标签字体大小，按需调整 */
            font-weight: 500 !important;
        }

        /* 输入框内的文字（占位符+用户输入的内容） */
        .stExpander div[data-testid="stTextInput"] > div {
            min-height: 70px !important; /* 调整输入框高度，数字越大框越高 */
            width: 100% !important; /* 保持100%撑满，也可设固定宽度如800px */
        }
        .stExpander div[data-testid="stTextInput"] input {
            font-size: 22px !important; /* 输入文字大小，自由调整 */
            padding: 22px 20px !important; /* 上下内边距，配合外层高度 */
            border-radius: 12px !important;
            height: 100% !important;
            border: 1px solid #e5e7eb !important; /* 可选：加边框更美观 */
        }

        .stExpander button[data-testid="stBaseButton-secondary"] {
            /* 核心：调整内边距 = 控制按钮整体大小 */
            padding: 20px 24px !important;
            /* 按钮高度（可选，和padding二选一即可） */
            min-height: 50px !important;
            /* 按钮圆角（保持和你界面风格一致） */
            border-radius: 12px !important;
            /* 宽度：100%撑满 / 固定像素 都可以 */
            width: 100% !important;
            background-color: #2158c4 !important; /* 按钮底色（可选） */
        }
        .stExpander button[data-testid="stBaseButton-secondary"] * {
            font-size: 22px !important;  /* 文字大小，自由调整 */
            font-weight: 500 !important; /* 文字粗细 */
            color: #ffffff
        }
        /* ====== 11. 数据表格 ====== */
        div[data-testid="stDataFrame"] {
            background-color: #F9F8F5 !important;
            border: 1px solid #DCD7CC !important;
            border-radius: 0px !important;
            padding: 0;
        }
        div[data-testid="stDataFrame"] [role="grid"] {
            background-color: #F9F8F5 !important;
        }
        div.element-container:has(div[data-testid="stDataFrame"]) {
            background-color: transparent !important;
            border: none !important;
            padding: 0 !important;
        }
        div[data-testid="stDataFrame"],
        div[data-testid="stDataFrame"] > div,
        div[data-testid="stDataFrame"] [data-testid="stWidgetLabel"] {
            background-color: transparent !important;
            border-radius: 0px !important;
        }

        /* ====== 12. JSON 展示：暗色终端风（修改版，带字体大小） ====== */
        div[data-testid="stJson"] {
            background-color: #1C1A17 !important;
            border: none !important;
            border-radius: 0px !important;
            padding: 16px !important;
            font-size: 16px !important; /* 父级基础字体大小 */
        }

        /* 关键：强制覆盖JSON内所有文字的大小（解决嵌套不生效问题） */
        div[data-testid="stJson"] span,
        div[data-testid="stJson"] div,
        div[data-testid="stJson"] div span,
        div[data-testid="stJson"] div div {
            font-size: 20px !important; /* ✅ 这里改你想要的大小：14/16/18px */
            color: #DCD7CC !important;
            font-family: 'JetBrains Mono', monospace !important;
        }

        div[data-testid="stJson"] > div {
            background-color: #1C1A17 !important;
        }

        /* ====== 13. 导出按钮：幽灵风 ====== */
        .stDownloadButton button {
            background: transparent !important;
            color: #1C1A17 !important;
            border: 1px solid #DCD7CC !important;
            border-radius: 0px !important;
            height: 48px !important;
            font-family: 'JetBrains Mono', monospace !important;
            font-size: 30px !important;
            font-weight: 500 !important;
            letter-spacing: 0.15em !important;
            text-transform: uppercase !important;
            width: 100% !important;
            transition: border-color 0.2s ease, color 0.2s ease;
        }
        .stDownloadButton button:hover {
            border-color: #D9381E !important;
            color: #D9381E !important;
        }

        /* ====== 15. 提示/标签 ====== */
        /* ====== 补充：修改 st.info/提示条的字体大小 ====== */
        div[data-testid="stAlert"] {
            font-size: 25px !important;
        }
        .film-tag {
            display: inline-flex;
            align-items: center;
            gap: 6px;
            padding: 3px 10px;
            border: 1px solid #DCD7CC;
            font-family: 'JetBrains Mono', monospace;
            font-size: 10px;
            font-weight: 500;
            letter-spacing: 0.1em;
            text-transform: uppercase;
            color: #6B6862;
            background: #F9F8F5;
        }
        .film-tag-accent {
            border-color: #D9381E;
            color: #D9381E;
            background: rgba(217, 56, 30, 0.05);
        }

        /* ====== 16. 分隔线 ====== */
        hr {
            border: none !important;
            height: 1px !important;
            background: #DCD7CC !important;
            margin: 3rem 0 !important;
        }

        /* ====== 17. st.success / error / info ====== */
        div[data-testid="stAlert"] {
            border: 1px solid #DCD7CC !important;
            border-radius: 0px !important;
            box-shadow: none !important;
            font-family: 'Work Sans', sans-serif !important;
            /* ✅ 核心：修改提示框整体字体大小 */
            font-size: 30px !important;
            line-height: 2 !important;
        }
        div[data-testid="stAlert"] p,
        div[data-testid="stAlert"] span,
        div[data-testid="stAlert"] div {
            font-size: 25px !important;
        }
        /* 控制提示框内加粗文字的大小（AI自动推荐视觉风格：【赛博朋克】） */
        div[data-testid="stAlert"] strong {
            font-size: 28px !important;
            color: #D9381E !important;
        }

        /* ====== 控制 help 悬浮提示文字大小 ====== */
        div[data-baseweb="tooltip"],
        div[data-baseweb="tooltip"] *,
        div[role="tooltip"],
        div[role="tooltip"] * {
            font-size: 20px !important;  /* ✅ 这里改你想要的大小 */
            line-height: 2 !important; /* 可选：调整行高，防止文字挤在一起 */
            color: #1C1A17 !important;   /* 可选：修改文字颜色，和你的界面风格统一 */
        }

        /* ====== 18. 滚动条 ====== */
        ::-webkit-scrollbar {
            width: 8px;
        }
        ::-webkit-scrollbar-track {
            background: #F4F1EB;
        }
        ::-webkit-scrollbar-thumb {
            background: #DCD7CC;
        }
        ::-webkit-scrollbar-thumb:hover {
            background: #6B6862;
        }

        /* ====== 20. 全局链接样式 ====== */
        a {
            color: #D9381E !important;
            text-decoration: none !important;
            transition: opacity 0.2s ease;
        }
        a:hover {
            opacity: 0.7;
        }

        /* ====== 21. Markdown 文字段落 ====== */
        .stMarkdown p {
            line-height: 1.8 !important;
            color: rgba(28, 26, 23, 0.8) !important;
            font-size: 25px !important;
        }
        .stMarkdown strong, .stMarkdown b {
            color: #1C1A17 !important;
            font-weight: 700 !important;
        }
        /* ====== 22. 修改内容 ====== */
        /* 确保角色图容器有间距，图片不溢出 */
        [data-testid="column"] img {
            border-radius: 8px !important; /* 给原画加点圆角更高级 */
            object-fit: contain !important; /* 确保图片完整显示，不裁剪 */
            margin-bottom: 15px !important;
            border: 1px solid #DCD7CC; /* 增加边框感 */
        }


        /* ====== 9. 下拉选择框 (缩小版) ====== */
        div[data-testid="stSelectbox"] label p {
            font-size: 16px !important;
            margin-bottom: 4px !important;
            line-height: 1.2 !important;
        }

        /* 选框外壳 */
        div[data-testid="stSelectbox"] div[data-baseweb="select"] > div {
            min-height: 42px !important;   /* 稍微增加一点高度 */
            height: 42px !important;
            padding: 0px 12px !important;  /* 取消上下内边距，靠行高居中 */
            display: flex !important;
            align-items: center !important; /* 强制垂直居中 */
            border-radius: 8px !important;
        }

        /* 内部文字容器 */
        div[data-testid="stSelectbox"] div[data-baseweb="select"] [data-testid="stMarkdownContainer"] p,
        div[data-testid="stSelectbox"] div[data-baseweb="select"] span,
        div[data-testid="stSelectbox"] div[role="combobox"] {
            font-size: 16px !important;
            line-height: 42px !important;  /* 关键：行高 = 容器高度，文字会完美垂直居中 */
            height: 42px !important;
            margin: 0 !important;
            padding: 0 !important;
        }

        /* 下拉箭头位置微调 */
        div[data-testid="stSelectbox"] svg {
            width: 18px !important;
            height: 18px !important;
            top: 2px !important; /* 调整箭头位置防止偏移 */
        }

        /* 展开后的选项列表 */
        div[data-baseweb="popover"] li * {
            font-size: 16px !important;
            line-height: 2 !important; /* 选项间距保持舒适 */
        }
    </style>
    """, unsafe_allow_html=True)



def get_client():
    target_key = st.session_state.get("active_text_key", API_KEY)
    return openai.OpenAI(api_key=target_key, base_url=BASE_URL, timeout=90)

SHOT_LEVEL_MAP = {
"Sparse": (1, 3),      # 新增：对应下拉选项 Sparse
    "Few": (4, 6),
    "Fewer": (4, 6),       # 保留兼容
    "Normal": (7, 9),
    "More": (10, 12),
    "Many": (13, 15),
    "Extensive": (13, 15) # 新增：对应下拉选项 Extensive
}

# ===================== Utility Functions =====================
def clean_json(s: str) -> dict:
    try:
        s = re.sub(r"```json|```", "", s).strip()
        match = re.search(r"(\{.*\}|\[.*\])", s, re.DOTALL)
        if match:
            data = json.loads(match.group(1))
            return {"data": data} if isinstance(data, list) else data
        return {}
    except:
        return {}

def generate_character_design(gender: str, age: str, appearance: str, clothing: str, feature: str, style: str) -> str:
    current_key = st.session_state.get("active_img_key", IMAGE_API_KEY)
    current_model = st.session_state.get("active_model", IMAGE_MODEL)
    try:
        prompt = (
            f"full body character portrait, front view, facing camera, 1 person only, solo, "
            f"character fills the entire vertical height of the image, top of the head touches the top edge, feet touch the bottom edge, "
            f"tight framing, no empty space above or below, no negative space, full height shot, "
            f"gender: {gender}, strictly {gender} features, no opposite gender characteristics, "
            f"age: {age}, facial features strictly match the age, appearance fully consistent with the described age, no age inconsistency, "
            f"character, {feature}, {appearance}, "
            f"wearing {clothing}, "
            f"pure solid white background, empty studio background, no objects, no clutter, no text, no annotations, "
            f"{style}, photorealistic, "
            f"cinematic lighting, sharp focus, 8k resolution"
        )

        payload = {
            "model": current_model,
            "prompt": prompt,
            "size": "2k",
            "n": 1,
            "response_format": "url"
        }

        url = f"{IMAGE_BASE_URL}/images/generations"
        headers = {
            "Authorization": f"Bearer {current_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(url, json=payload, headers=headers, timeout=180)
        response.raise_for_status()
        res_json = response.json()
        return res_json["data"][0]["url"]

    except Exception as e:
        st.error(f"Character image generation failed: {str(e)}")
        return "https://static.streamlit.io/examples/dark-gray.png"

def generate_storyboard_frame(prompt: str, ref_images: list[str] = None) -> str:
    current_key = st.session_state.get("active_img_key", IMAGE_API_KEY)
    current_model = st.session_state.get("active_model", IMAGE_MODEL)

    try:
        valid_refs = [r for r in ref_images if r and r.startswith("http")] if ref_images else []

        payload = {
            "model": current_model,
            "prompt": prompt,
            "size": "2k",
            "n": 1,
            "response_format": "url",
            # 2. 官方标准：直接放在 image 数组
            "image": ref_images if ref_images else [],
            # 3. 开启序列生成逻辑 (如果是多张分镜一起生成)
            "sequential_image_generation": "auto",
            "sequential_image_generation_options": {
                "max_images": 1 # 如果你依然想单张调用，也要保证 image 字段正确
            }
        }

        if ref_images:
            valid_refs = [r for r in ref_images if r and r.startswith("http")]
            if valid_refs:
                payload["extra"] = {
                    "reference_images": valid_refs,
                    "reference_strength": 0.99,
                    "reference_mode": "character",
                    "reference_color_mode": "reference"
                }

        url = f"{IMAGE_BASE_URL}/images/generations"
        headers = {
            "Authorization": f"Bearer {current_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(url, json=payload, headers=headers, timeout=180)
        response.raise_for_status()
        res_json = response.json()
        return res_json["data"][0]["url"]

    except Exception as e:
        st.error(f"Storyboard frame generation failed: {str(e)}")
        return "https://static.streamlit.io/examples/dark-gray.png"

# ===================== Get Character Reference Images =====================
def get_character_ref_images(char_names: list, char_design_lib: list) -> list:
    st.write(f"调试：当前镜头尝试匹配的角色名: {char_names}")
    ref_images = []
    if not char_names or not char_design_lib:
        return ref_images
    # 预处理库，方便匹配
    lookup = {str(c.get("Name")).strip().lower(): c.get("Design Sheet") for c in char_design_lib}
    for name in char_names:
        clean_name = str(name).strip().lower()
        if clean_name in lookup and lookup[clean_name]:
            ref_images.append(lookup[clean_name])

    return ref_images

# ===================== Skill 1: Script Parsing Agent =====================
class DirectorAgent:
    def parse_script(self, script: str, role_text: str, shot_level: str) -> Dict:
        min_shot, max_shot = SHOT_LEVEL_MAP[shot_level]
        prompt_template = """
            Character Description: [[ROLE_TEXT]]
            Script: [[SCRIPT_CONTENT]]

            [Strict Task: Structured Script Parsing]
            Extract all information as required, **no fabrication**, output pure JSON, **all fields required**:
            1. Character List: must include all fields:
              - Name
              - Gender
              - Age
              - Clothing
              - Appearance
              - Core Features
            2. Scene: specific environment + details
            3. Core Actions: continuous actions of all characters
            4. Dialogue List: extract all lines in order with character names
            5. Mood & Atmosphere: character emotions + overall scene mood
            6. Plot Pace: fast / medium / slow
            7. Visual Style: cinematic / anime / ancient / cyberpunk / heartwarming, etc.
            8. Fixed Lighting: natural / top / side / soft, etc.
            9. Fixed Color Tone: warm / cold / vintage / high saturation, etc.
            10. Recommended Shot Count: between [[MIN]] and [[MAX]]

            Output Format (strict JSON, no extra text):
            {
                "Character List": [{"Name":"","Gender":"","Age":"","Clothing":"","Appearance":"","Core Features":""}],
                "Dialogue List": [{"Character":"","Dialogue":""}],
                "Scene":"",
                "Scene Details":"",
                "Core Actions":"",
                "Mood & Atmosphere":"",
                "Plot Pace":"",
                "Recommended Visual Style":"",
                "Fixed Lighting":"",
                "Fixed Color Tone":"",
                "Recommended Shot Count": 0
            }"""

        prompt = prompt_template.replace("[[ROLE_TEXT]]", role_text) \
                     .replace("[[SCRIPT_CONTENT]]", script) \
                     .replace("[[MIN]]", str(min_shot)) \
                     .replace("[[MAX]]", str(max_shot))

        try:
            client = get_client()
            res = client.chat.completions.create(
                model=AGENT_TYPE,
                messages=[{"role":"user","content":prompt}]
            )
            content = res.choices[0].message.content
            result = clean_json(content)

            if not result:
                st.error(f"Script parsing failed, AI returned content: {content[:100]}...")
                return {}

            shot_num = result.get("Recommended Shot Count", min_shot)
            result["Recommended Shot Count"] = max(min_shot, min(int(shot_num), max_shot))
            return result
        except Exception as e:
            st.error(f"An exception occurred during script parsing: {str(e)}")
            return {}

# ===================== Skill 2+3: Storyboard Generation + Consistency Control =====================
class CinematographyAgent:
    def generate_shots(self, parse_info: Dict, shot_count: int) -> List[Dict]:
        parse_str = json.dumps(parse_info, ensure_ascii=False, indent=2)
        prompt = f"""
            [Script Analysis]: {parse_str}
            [Requirement]: Generate {shot_count} professional storyboard shots.
            Rule 1 (Character Consistency):
                - Each shot must list Character Names clearly.
                - Character appearance, clothing, features must stay consistent.
            Rule 2: Shot types cycle: wide, medium, close-up, extreme close-up.
            Rule 3: Unified character, scene, lighting, color across all shots.
            Rule 4: Dialogue must be filled accurately from script.
            Rule 5: Shot numbers 1~N in order.
            Rule 6: Camera Movement Rules
                - Dialogue/emotion: static / slight push
                - Walking/action: tracking / dolly
                - Environment: pan
                - Details: push + close-up
            Rule 7: Transition Rules
                - Scene change: fade in/out
                - Smooth mood: dissolve
                - Fast pace: cut
                - Start: fade in, end: fade out
            Rule 8: Duration Rules
                - Close-up: 1~3s
                - Medium close-up: 3~5s
                - Medium shot: 5~7s
                - Wide/long shot: 7~9s

            Output pure JSON array only:
            [{{
                "Shot Number": 1,
                "Shot Type": "wide/medium/close-up/extreme close-up",
                "Visual Description": "detailed scene with character names and features",
                "Character Dialogue": "exact lines from script",
                "Characters Appearing": ["Name"],
                "Duration (s)": number,
                "Camera Movement": "static/push/pull/pan/dolly/tracking",
                "Transition": "fade in/fade out/dissolve/cut"
            }}]
            """
        client = get_client()
        if not client:
            return []
        try:
            res = client.chat.completions.create(
                model=AGENT_TYPE,
                messages=[{"role":"user","content":prompt}],
                temperature=0.2
            )
            data = clean_json(res.choices[0].message.content)
            return data.get("data", [])[:shot_count]
        except Exception as e:
            st.error(f"Storyboard generation failed: {str(e)}")
            return []

# ===================== Skill 4: Seedance Prompt Generation =====================
class TechTransAgent:
    def generate_seedance_prompt(self, shot: Dict, consistency_lock: str, style: str, char_info: str, pace: str, light: str, color: str) -> str:

        instruction = f"""
            FIRST RULE: The character in the final image MUST be a 1:1 copy of the character reference sheet. Do NOT alter any part of the character's design.
            [CHARACTER DESCRIPTIONS]:{consistency_lock}
            [CURRENT SHOT TO TRANSLATE]:Visual: {shot['Visual Description']}
            Shot Type: {shot['Shot Type']}
            LIGHTING: {light}
            COLOR TONE: {color}
            ART STYLE: {style}

            [STRICT RULES]:
            1. DO NOT just use character names. Replace names with their PHYSICAL DESCRIPTIONS from the character list.
              (e.g., instead of "Su Lao", write "a 70-year-old man with long white hair and glasses")
            2. Describe the characters' clothing specifically as defined in the list.
            3. Ensure the distinction between different characters is clear in the prompt.
            4. Style: {style}, Lighting: {light}, Color: {color}.

            OUTPUT ONLY THE ENGLISH PROMPT..
            """

        try:
            client = get_client()
            res = client.chat.completions.create(
                model=AGENT_TYPE,
                messages=[
                    {"role": "system", "content": "You are a professional Hollywood concept artist."},
                    {"role": "user", "content": instruction}
                ],
                temperature=0.7
            )
            en_content = res.choices[0].message.content.strip()
            en_content = re.sub(r'^(Prompt:|DALL-E 3 Prompt:|Output:)', '', en_content, flags=re.IGNORECASE).strip()

        except:
            en_content = f"Cinematic photography, {style}, {shot['Visual Description']}, high quality"

        constraint = (
            "FIRST PRIORITY: Character must be EXACTLY identical to reference. "
            "No changes to face, hair, clothing, mechanical parts, color, scars. "
            "Only background and lighting may vary."
        )

        prompt = f"{en_content}, {constraint}, single frame, cinematic still, shot on 35mm lens, 8k resolution, realistic textures --ar 16:9"
        return re.sub(r'[\u4e00-\u9fa5]', '', prompt)

# ===================== Orchestrator =====================
class StoryboardOrchestrator:
    def __init__(self):
        self.director = DirectorAgent()
        self.cinema = CinematographyAgent()
        self.trans = TechTransAgent()

    def run(self, script: str, role_text: str, shot_level: str, model_name: str):
        parse_result = self.director.parse_script(script, role_text, shot_level)
        if not parse_result:
            return [], "", {}

        char_desc_list = []
        for c in parse_result.get("Character List", []):
            desc = (
                f"{c['Name']}: (Gender: {c.get('Gender','')}, Age: {c.get('Age','')}, "
                f"Appearance: {c.get('Appearance','')}, Wearing: {c.get('Clothing','')}, Features: {c.get('Core Features','')})"
            )
            char_desc_list.append(desc)

        consistency_lock = " | ".join(char_desc_list) + f" | Global Environment: {parse_result['Scene']} | Lighting: {parse_result['Fixed Lighting']}"
        shot_count = parse_result["Recommended Shot Count"]
        shots = self.cinema.generate_shots(parse_result, shot_count)

        final_shots = []
        char_design_lib = parse_result.get("Character Design Library", [])

        for idx, shot in enumerate(shots[:shot_count], 1):
            current_char_names = shot.get("Characters Appearing", [])
            relevant_chars_info = [
                f"{c['Name']}({c['Appearance']}, wearing {c['Clothing']})"
                for c in parse_result.get("Character List", [])
                if c['Name'] in current_char_names
            ]
            char_info_str = " | ".join(relevant_chars_info)

            en_prompt = self.trans.generate_seedance_prompt(
                shot=shot,
                consistency_lock=consistency_lock,
                style=parse_result.get("Recommended Visual Style", "Cinematic"), # 传给 style
                char_info=char_info_str, # 传给 char_info
                pace=parse_result.get("Plot Pace", "Normal"),
                light=parse_result.get("Fixed Lighting", "Natural"),
                color=parse_result.get("Fixed Color Tone", "Standard")
            )

            current_char_names = shot.get("Characters Appearing", [])
            char_urls = get_character_ref_images(current_char_names, char_design_lib)

            try:
                image_url = generate_storyboard_frame(en_prompt, ref_images=char_urls)
            except:
                image_url = "https://static.streamlit.io/examples/dark-gray.png"

            final_shots.append({
                "Shot": idx,
                "Type": shot.get("Shot Type", "Wide"),
                "Duration": shot.get("Duration (s)", 3),
                "Visual": shot.get("Visual Description", ""),
                "Dialogue": shot.get("Character Dialogue", "None"),
                "Movement": shot.get("Camera Movement", "Static"),
                "Transition": shot.get("Transition", "Fade In"),
                "Prompt": en_prompt,
                "Key Frame": image_url,
                "Reference Images": char_urls,
                "Reference Characters": current_char_names
            })

        return final_shots, parse_result["Recommended Visual Style"], parse_result

    def regenerate_single_shot(self,
                  shot_index: int,
                  total_shots: List[Dict],
                  original_script: str,
                  role_text: str,
                  parse_data: Dict,
                  model_name: str,
                  user_modify_requirement: str = "") -> Dict:
        prev_shot = total_shots[shot_index-2] if shot_index > 1 else None
        next_shot = total_shots[shot_index] if shot_index < len(total_shots) else None
        current_shot = total_shots[shot_index-1]
        if not current_shot:
            st.error("Current shot data empty, cannot regenerate!")
            return {}

        context_prompt = ""
        if prev_shot:
            context_prompt += f"[Previous Shot]: {json.dumps(prev_shot, ensure_ascii=False)}\n"
        if next_shot:
            context_prompt += f"[Next Shot]: {json.dumps(next_shot, ensure_ascii=False)}\n"

        consistency_lock = (
            f"Fixed Characters: {'; '.join([c['Name']+'-'+c['Clothing']+'-'+c['Appearance'] for c in parse_data['Character List']])} | "
            f"Fixed Scene: {parse_data['Scene']} | Fixed Lighting: {parse_data['Fixed Lighting']} | "
            f"Fixed Color Tone: {parse_data['Fixed Color Tone']} | Fixed Style: {parse_data['Recommended Visual Style']}"
        )

        prompt = f"""
    [Global Fixed Settings]
    {consistency_lock}
    Original Script: {original_script}
    [Shot Context]
    {context_prompt}
    [Current Shot]: {json.dumps(current_shot, ensure_ascii=False)}
    [User Modification Request]: {user_modify_requirement.strip()}

    [Regeneration Rules]
    1. Follow user request strictly.
    2. Auto-detect characters in scene.
    3. Use original dialogue.
    4. Auto duration by shot type:
       - Close-up: 1~3s
       - Medium close-up: 3~5s
       - Medium shot: 5~7s
       - Wide/long shot: 7~9s
    5. Output pure JSON only.

    [Output Format]
    {{
        "Shot Type": "",
        "Visual Description": "",
        "Character Dialogue": "",
        "Camera Movement": "",
        "Transition": "",
        "Characters Appearing": [],
        "Duration (s)": 0
    }}
    """
        try:
            client = get_client()
            res = client.chat.completions.create(
                model=AGENT_TYPE,
                messages=[{"role":"user","content":prompt}],
                temperature=0.3
            )
            new_shot_raw = clean_json(res.choices[0].message.content)

            current_char_names = new_shot_raw.get("Characters Appearing", current_shot["Reference Characters"])
            char_urls = get_character_ref_images(current_char_names, parse_data.get("Character Design Library", []))

            new_shot = {
                "Shot": shot_index,
                "Type": new_shot_raw.get("Shot Type", current_shot["Type"]),
                "Duration": new_shot_raw.get("Duration (s)", current_shot["Duration"]),
                "Visual": new_shot_raw.get("Visual Description", current_shot["Visual"]),
                "Dialogue": new_shot_raw.get("Character Dialogue", current_shot["Dialogue"]),
                "Movement": new_shot_raw.get("Camera Movement", current_shot["Movement"]),
                "Transition": new_shot_raw.get("Transition", current_shot["Transition"]),
                "Reference Images": char_urls,
                "Reference Characters": current_char_names
            }

            current_char_names = new_shot_raw.get("Characters Appearing", current_shot["Reference Characters"])
            relevant_chars_info = [
                f"{c['Name']}({c['Appearance']}, wearing {c['Clothing']})"
                for c in parse_data.get("Character List", [])
                if c['Name'] in current_char_names
            ]
            char_info_str = " | ".join(relevant_chars_info)

            new_shot["Prompt"] = self.trans.generate_seedance_prompt(
                {"Shot Type": new_shot["Type"], "Visual Description": new_shot["Visual"], "Camera Movement": new_shot["Movement"]},
                consistency_lock,
                parse_data["Recommended Visual Style"],char_info_str, parse_data["Plot Pace"],
                parse_data["Fixed Lighting"], parse_data["Fixed Color Tone"]
            )

            try:
                new_img_url = generate_storyboard_frame(new_shot["Prompt"], ref_images=char_urls)
                if new_img_url:
                    new_shot["Key Frame"] = new_img_url
                else:
                    new_shot["Key Frame"] = current_shot.get("Key Frame", "https://static.streamlit.io/examples/dark-gray.png")
                    st.warning(f"Shot {shot_index} image failed, kept original.")
            except Exception as e:
                new_shot["Key Frame"] = current_shot.get("Key Frame", "https://static.streamlit.io/examples/dark-gray.png")
                st.error(f"Image generation error: {str(e)}, kept original.")

            return new_shot
        except Exception as e:
            st.error(f"Single shot regeneration failed: {str(e)}")
            return current_shot

# ===================== Export Tools =====================
def export_txt(shots):
    lines = ["[AI Painting Storyboard]\n"]
    for s in shots:
        lines.append(f"Shot {s['Shot']} | {s['Type']} | {s['Duration']}s")
        lines.append(s['Prompt'])
        lines.append("-"*50)
    return "\n".join(lines)

def export_word(shots, style):
    doc = Document()
    doc.add_heading(f'Short Drama Storyboard ({style})', 0)
    for s in shots:
        doc.add_paragraph(f"Shot {s['Shot']} | {s['Type']} | {s['Duration']}s")
        doc.add_paragraph(f"Visual: {s['Visual']}")
        doc.add_paragraph(f"Dialogue: {s['Dialogue']}")
        doc.add_paragraph(f"Movement: {s['Movement']} | Transition: {s['Transition']}")
        doc.add_paragraph("-"*40)
    return doc

def export_excel(shots):
    data = []
    for s in shots:
        data.append({
            "Shot No.": s["Shot"],
            "Shot Type": s["Type"],
            "Duration (s)": s["Duration"],
            "Camera Movement": s["Movement"],
            "Transition": s["Transition"],
            "Visual Description": s["Visual"],
            "Dialogue": s["Dialogue"],
            "Seedance Prompt": s["Prompt"]
        })
    df = pd.DataFrame(data)
    buffer = io.BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        df.to_excel(writer, index=False)
    buffer.seek(0)
    return buffer

# ===================== Page 1: Home =====================
def home_page():
    apply_ultra_tech_theme()
    st.markdown("<h1>🎬 AI STORYBOARD SYSTEM</h1>", unsafe_allow_html=True)

    st.markdown("""
    <div class="tech-card">
        <h3 style="margin:0">🌟 SYSTEM INITIATION / System Features</h3>
        <p style="font-size:22px; margin-top:15px; color:#ccc;">
            This is a professional AI-powered short drama storyboard generation tool. Build a cinematic universe in 3 steps:<br>
            1. <b>IDENTITY</b>: Describe character appearance, clothing, and facial features<br>
            2. <b>NARRATIVE</b>: Inject the soul of your short drama script<br>
            3. <b>SYNTHESIS</b>: Automatically decompose professional storyboard scripts
        </p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<h2>✨ CORE CAPABILITIES / Core Features</h2>", unsafe_allow_html=True)

    c1, c2 = st.columns(2)
    with c1:
        st.markdown(f'<div class="tech-card" style="font-size: 25px;">✅ <b>STYLING</b>: AI automatically analyzes scripts and recommends optimal visual styles</div>', unsafe_allow_html=True)
        st.markdown(f'<div class="tech-card" style="font-size: 25px;">✅ <b>PROMPTS</b>: One-click generation of AI painting prompt library</div>', unsafe_allow_html=True)
    with c2:
        st.markdown(f'<div class="tech-card" style="font-size: 25px;">✅ <b>CINEMA</b>: Professional storyboard generation (shot type / camera movement / duration)</div>', unsafe_allow_html=True)
        st.markdown(f'<div class="tech-card" style="font-size: 25px;">✅ <b>REGEN</b>: Precise single-shot regeneration / logic correction</div>', unsafe_allow_html=True)

    st.markdown(f"""
        <div style="background: rgba(0,255,0,0.05); border: 1px dashed #00FF00; padding:20px; text-align:center; color:#00FF00; font-family:'Orbitron'; font-size: 25px;">
            >>> READY FOR INPUT ... SELECT "CREATOR" FROM SIDEBAR
        </div>
        """, unsafe_allow_html=True)

# ===================== Page 2: Creation & Storyboarding =====================
def input_page():
    if "shots" not in st.session_state:
          st.session_state.shots = []
    apply_ultra_tech_theme()
    st.markdown("<h1 style='margin:0; padding:0; line-height:1; margin-bottom:-10px;'>✍️ CREATION PANEL</h1>", unsafe_allow_html=True)

    with st.container():
        st.markdown("## 👤 CHARACTER SETTING")
        role_text = st.text_area("", placeholder="DESCRIPTOR: Lao Chen, 58 years old, gray hair...", height=200, label_visibility="collapsed")

        st.markdown("<br>", unsafe_allow_html=True)

        st.markdown("## 📖 SCRIPT CONTENT")
        script = st.text_area("", placeholder="SOURCE SCRIPT: Paste script content here...", height=400, label_visibility="collapsed")

        st.markdown("<br>", unsafe_allow_html=True)

        st.markdown("## 🎞️ SHOT DENSITY")
        shot_level = st.selectbox("", options=["Sparse", "Few", "Normal", "Many", "Extensive"], index=2, help="Sparse: 1-3 shots | Few: 4-6 shots | Normal: 7-9 shots | Many: 10-12 shots | Extensive: 13-15 shots")

    st.markdown("<br>", unsafe_allow_html=True)

    if st.button("🚀 Generate Storyboard ", type="primary", use_container_width=True):
        if not script or not role_text:
            st.error("Please complete both character settings and script content!")
        else:
            if "current_seed" not in st.session_state:
                st.session_state.current_seed = random.randint(1, 10**9)

            with st.spinner("#### 🎬 Step 1: Parsing Script..."):
                parse_result = st.session_state.orch.director.parse_script(script, role_text, shot_level)
                if not parse_result:
                    st.error("Parsing failed"); st.stop()

                st.session_state.parse = parse_result
                st.session_state.ai_style = parse_result.get("Recommended Visual Style", "Cinematic Realism")
                st.session_state.script = script
                st.session_state.role_text = role_text

            # --- Character Image Generation ---
            char_list = parse_result.get("Character List", [])
            char_designs = []
            if char_list:
                st.info(f"🎭 Generating character design sheets ({len(char_list)} characters)...")
                for char in char_list:
                    char_url = generate_character_design(
                        gender=char.get("Gender", ""),
                        age=char.get("Age", ""),
                        appearance=char.get("Appearance", ""),
                        clothing=char.get("Clothing", ""),
                        feature=char.get("Core Features", ""),
                        style=st.session_state.ai_style
                    )
                    char_designs.append({"Name": char["Name"], "Design Sheet": char_url})
                st.session_state.parse["Character Design Library"] = char_designs

            with st.spinner("#### 🎞️ Step 2: Generating Storyboard Framework..."):
                consistency_lock = f"Style: {st.session_state.ai_style} | Scene: {parse_result['Scene']}"
                shot_count = parse_result["Recommended Shot Count"]
                raw_shots = st.session_state.orch.cinema.generate_shots(parse_result, shot_count)

            st.success(f"✅ Framework ready, starting storyboard rendering...")
            final_shots = []
            progress_bar = st.progress(0)

            for idx, shot in enumerate(raw_shots, 1):
                st.write(f"⏳ Rendering: Shot {idx}/{len(raw_shots)}...")

                # 1. 精准提取当前镜头出现的角色名字
                current_char_names = shot.get("Characters Appearing", [])

                # 2. 从解析出的角色列表中提取这些角色的完整物理描述，传给 prompt 生成器
                # 这样 GPT 才知道“苏老”到底长什么样
                relevant_chars_info = [
                    f"{c['Name']}({c['Appearance']}, wearing {c['Clothing']})"
                    for c in st.session_state.parse.get("Character List", [])
                    if c['Name'] in current_char_names
                ]
                char_info_str = " | ".join(relevant_chars_info)


                en_prompt = st.session_state.orch.trans.generate_seedance_prompt(
                    shot,
                    consistency_lock,
                    st.session_state.ai_style,
                    char_info_str,
                    parse_result["Plot Pace"],
                    parse_result["Fixed Lighting"],
                    parse_result["Fixed Color Tone"]
                )

                char_urls = get_character_ref_images(current_char_names, st.session_state.parse.get("Character Design Library", []))

                try:
                    img_url = generate_storyboard_frame(en_prompt, ref_images=char_urls)
                except:
                    img_url = None

                if not img_url:
                    img_url = "https://static.streamlit.io/examples/dark-gray.png"

                final_shots.append({
                    "Shot": idx,
                    "Type": shot.get("Shot Type", "Medium Shot"),
                    "Duration": shot.get("Duration (s)", 3),
                    "Visual": shot.get("Visual Description", ""),
                    "Dialogue": shot.get("Character Dialogue", "None"),
                    "Movement": shot.get("Camera Movement", "Static"),
                    "Transition": shot.get("Transition Effect", "Cut"),
                    "Prompt": en_prompt,
                    "Key Frame": img_url,
                    "Reference Images": char_urls,
                    "Reference Characters": [c["Name"] for c in st.session_state.parse.get("Character Design Library", [])]
                })
                st.session_state.shots = final_shots
                progress_bar.progress(idx / len(raw_shots))

            st.rerun()

    # ===================== Storyboard Results =====================
    st.markdown("---")
    st.title("🎥 Storyboard Results")
    st.caption("Collapsible storyboard preview with one-click single-shot regeneration")

    if "shots" in st.session_state and st.session_state.shots:
        shots = st.session_state.shots
        st.success(f"AI Recommended Visual Style: **{st.session_state.ai_style}**", icon="✨")

        if "parse" in st.session_state:
            with st.expander("📄 Script Structured Parsing", expanded=False):
                st.json(st.session_state.parse)
            char_data = st.session_state.parse.get("Character Design Library", [])
            if char_data:
                st.markdown("### 🎭 Character Design Sheets")
                cols = st.columns(4)
                for i, char in enumerate(char_data):
                    with cols[i % 4]:
                        if char.get("Design Sheet"):
                            caption_html = f"""
                            <div style="text-align: center; font-size: 25px; margin-top: 8px; color: #1C1A17;">
                                {char["Name"]}
                            </div>
                            """
                            st.image(char["Design Sheet"], width="stretch")
                            st.markdown(caption_html, unsafe_allow_html=True)
            else:
                st.info("💡 No character design sheets generated yet.")

        st.markdown("---")
        st.markdown("### 🎬 Full Storyboard Script Sequence")
        for idx, item in enumerate(st.session_state.shots):
            shot_num = item["Shot"]
            with st.expander(f"📽️ Shot {shot_num} | {item['Type']} | {item['Duration']}s", expanded=(idx==0)):

                info_col_left, info_col_right = st.columns([1, 1.2])

                with info_col_left:
                    st.markdown(f"**🎥 Camera Movement**：{item['Movement']}")
                    st.markdown(f"**🎞️ Transition Effect**：{item['Transition']}")
                    st.markdown(f"**🧬 Reference Character Sheets**：{', '.join(item.get('Reference Characters', []))}")
                    if "Reference Images" in item and item["Reference Images"]:
                        urls = [u for u in item["Reference Images"] if u and u.startswith("http")]
                        if urls:
                            st.markdown(f"Linked Character References: {len(urls)}")
                            ref_cols = st.columns(4)
                            for i, url in enumerate(urls):
                                ref_cols[i].image(url, width=150)

                with info_col_right:
                    st.markdown("**🎬 Scene Description**：")
                    st.markdown(f"{item['Visual']}")
                    st.markdown(f"**💬 Dialogue**：{item['Dialogue']}")

                st.markdown("---")
                col_preview, col_prompt = st.columns(2)

                with col_preview:
                    st.markdown("#### 🖼️ Shot Preview")
                    st.markdown(f"""
                    <div style="width:100%; height:400px; border-radius:8px; overflow:hidden; margin-bottom:8px;">
                        <img src="{item['Key Frame']}" style="width:100%; height:100%; object-fit:cover;">
                    </div>
                    <div style="text-align:center; font-size:20px; color:#6B6862; margin-top:5px;">
                        Shot {shot_num} Preview
                    </div>
                    """, unsafe_allow_html=True)

                with col_prompt:
                    st.markdown("#### ✏️ English Prompt (Seedance)")
                    st.markdown(f"""
                      <div style="height:400px; overflow-y:auto; border:1px solid #DCD7CC;
                          border-radius:0px; padding: 0px 8px; background:#F9F8F5;">
                          <p style="white-space:pre-wrap; word-break:break-word;
                              font-family:'JetBrains Mono',monospace;
                              font-size:22px !important;
                              line-height:1.6;
                              margin:0;">{item['Prompt']}</p>
                      </div>
                      """, unsafe_allow_html=True)

                mod_req = st.text_input(f"✏️ Modification for Shot {shot_num}", placeholder="e.g. Change to close-up side view...", key=f"req_{shot_num}")
                if st.button(f"🔄 Regenerate Shot {shot_num}", key=f"btn_{shot_num}", use_container_width=True):
                    with st.spinner(f"Regenerating Shot {shot_num}..."):
                        new_shot = st.session_state.orch.regenerate_single_shot(
                            shot_index=shot_num,
                            total_shots=st.session_state.shots,
                            original_script=st.session_state.script,
                            role_text=st.session_state.role_text,
                            parse_data=st.session_state.parse,
                            model_name="",
                            user_modify_requirement=mod_req
                        )
                        st.session_state.shots[idx] = new_shot
                        st.rerun()
    else:
        st.info("👆 Please fill in the script above and click 'Generate Storyboard in One Click'")

    # ===================== Full Storyboard Overview Table =====================
    st.markdown("---")
    st.title("📊 Full Storyboard Overview")
    st.caption("One-click copy, filter, and export all storyboard data")

    if "shots" in st.session_state and st.session_state.shots:
        col1, col2, col3 = st.columns(3)

        with col1:
            txt_data = export_txt(st.session_state.shots)
            st.download_button("📝 AI-Ready TXT File", txt_data, "Storyboard_AI_Ready.txt", use_container_width=True)

        with col2:
            doc = export_word(st.session_state.shots, st.session_state.ai_style)
            doc.save("temp.docx")
            with open("temp.docx", "rb") as f:
                st.download_button("📘 Report Word File", f, "Storyboard_Report.docx", use_container_width=True)

        with col3:
            excel_buffer = export_excel(st.session_state.shots)
            st.download_button("📊 Storyboard Excel File", excel_buffer, "Storyboard.xlsx", use_container_width=True)

    if st.session_state.shots:
        table_data = []
        for shot in st.session_state.shots:
            table_data.append({
                "Shot No.": shot["Shot"],
                "Type": shot["Type"],
                "Duration (sec)": shot["Duration"],
                "Camera Movement": shot["Movement"],
                "Transition Effect": shot["Transition"],
                "Visual Description": shot["Visual"],
                "Dialogue": shot["Dialogue"],
                "Seedance Prompt": shot["Prompt"]
            })

        df = pd.DataFrame(table_data)
        st.dataframe(
            df,
            use_container_width=True,
            height=450,
            column_config={
                "Seedance Prompt": st.column_config.TextColumn(width=500)
            }
        )
    else:
        st.info("👆 Please generate storyboard first to view overview")

# ===================== Main Program =====================
def main():
  st.set_page_config(layout="centered", initial_sidebar_state="collapsed")

  # --- 侧边栏 API 设置区 ---
  with st.sidebar:
      st.markdown("### ⚙️ Settings")
      user_text_key = st.text_input("Custom OpenAI Key", type="password", placeholder="Leave blank to use default")
      user_img_key = st.text_input("Custom Image Key", type="password", placeholder="Leave blank to use default")

      # 2. 模型选择窗口
      model_options = {
          "Doubao 4.0": "doubao-seedream-4-0-250828",
          "Doubao 4.5": "doubao-seedream-4-5-251128",
          "Doubao 5.0": "doubao-seedream-5-0-lite-260128"
      }
      selected_model_label = st.selectbox("Select Image Model", options=list(model_options.keys()))
      st.session_state.active_model = model_options[selected_model_label]

      st.markdown("---")
      if user_text_key:
          st.caption("✅ Using Custom OpenAI Key")
      else:
          st.caption("ℹ️ Using System Default OpenAI Key")

      if user_img_key:
          st.caption("✅ Using Custom Image Key")
      else:
          st.caption("ℹ️ Using System Default Image Key")

  # 将获取到的 Key 存入 session_state 供全局调用
  # 逻辑：如果用户填了就用用户的，没填就用代码里的常量
  st.session_state.active_text_key = user_text_key if user_text_key else API_KEY
  st.session_state.active_img_key = user_img_key if user_img_key else IMAGE_API_KEY

  if "shots" not in st.session_state:
      st.session_state.shots = []
  if "orch" not in st.session_state:
      st.session_state.orch = StoryboardOrchestrator()
  pages = [
      st.Page(home_page, title="Home", icon="🏠"),
      st.Page(input_page, title="Create", icon="📝"),
  ]
  nav = st.navigation(pages, position="sidebar")
  nav.run()

if __name__ == "__main__":
    main()

Overwriting storyboard_agent.py


In [ ]:
# 强制杀死所有旧 ngrok 进程，清空隧道占用
!pkill -f ngrok

# 重新安装/刷新 pyngrok（防止缓存异常）
!pip install -U pyngrok

In [4]:
# ===================== 3. Colab运行Streamlit（ngrok版） =====================
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3BzPAxKOu1H5oA78LnVRWWzk9vH_4K3G18sc3wV1tThgPcfpa"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 启动Streamlit
!streamlit run storyboard_agent.py --server.port 8051 --server.headless true &>/dev/null&

# 启动ngrok穿透
public_url = ngrok.connect(8051)
print(f"✅ 访问地址：{public_url}")

✅ 访问地址：NgrokTunnel: "https://vince-unsidling-cherelle.ngrok-free.dev" -> "http://localhost:8051"
